In [1]:
!pip install wbgapi pandas plotly kaleido

import wbgapi as wb
import pandas as pd
import plotly.express as px

# Same European countries as Project 1
europe = ['DEU','FRA','ITA','ESP','POL','NLD','BEL','AUT',
          'SWE','CZE','HUN','PRT','GRC','ROU','SVK','FIN']

# Export sectors data - Value added by sector as % of GDP
sectors = {
    'Manufacturing': 'NV.IND.MANF.ZS',
    'Services': 'NV.SRV.TOTL.ZS',
    'Agriculture': 'NV.AGR.TOTL.ZS'
}

data = {}
for sector, code in sectors.items():
    df = wb.data.DataFrame(code, economy=europe, time=range(2015, 2023))
    data[sector] = df

print("Data loaded!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 1.5 MB/s eta 0:00:00
Data loaded!


In [3]:
# Frey & Osborne automation risk by sector (from their 2017 paper)
# These are established numbers from the paper
automation_risk = {
    'Manufacturing': 0.70,  # 70% automation probability
    'Services': 0.52,
    'Agriculture': 0.55
}

# Get latest data (2021) for each country
rows = []
for sector, code in sectors.items():
    df = data[sector]
    # The original line below incorrectly tried to convert country codes in the index to integers.
    # df.index = df.index.str.replace('YR','').astype(int)
    # To get the data for 2021, we should select the 'YR2021' column.
    latest = df['YR2021'].dropna()

    for country, value in latest.items():
        rows.append({
            'country': country,
            'sector': sector,
            'gdp_share': value,
            'automation_risk': automation_risk[sector]
        })

df_all = pd.DataFrame(rows)

# Weighted automation risk per country
df_all['weighted_risk'] = df_all['gdp_share'] * df_all['automation_risk']
country_risk = df_all.groupby('country')['weighted_risk'].sum().reset_index()
country_risk.columns = ['country', 'total_risk_score']
country_risk = country_risk.sort_values('total_risk_score', ascending=False)

print(country_risk)

   country  total_risk_score
3      DEU         46.459061
2      CZE         46.366671
9      ITA         45.479034
0      AUT         44.951715
1      BEL         44.670893
10     NLD         44.548896
13     ROU         44.488787
4      ESP         44.478355
14     SVK         43.933722
6      FRA         43.922175
7      GRC         43.853545
12     PRT         43.734904
15     SWE         43.414612
8      HUN         43.301687
5      FIN         42.928370
11     POL         42.700604


In [5]:
fig_map = px.choropleth(
    country_risk,
    locations='country',
    color='total_risk_score',
    color_continuous_scale='RdYlGn_r',
    scope='europe',
    title='AI Automation Risk Score by EU Country (2021)'
)
fig_map.show()

# Save as image
# The fig.write_image() function is currently incompatible with the installed Plotly and Kaleido versions.
# To enable image saving, consider updating Plotly to version 6.1.1 or greater, or downgrading Kaleido to version 0.2.1 in your package installation cell.
# fig_map.write_image("ai_risk_map.png")

In [8]:
# How to predict the actual unemployment?

import statsmodels.api as sm  # Import statsmodels library

unemp = wb.data.DataFrame(
    'SL.UEM.TOTL.ZS',
    economy=europe,
    time=range(2021, 2022)
)

# Correctly select the unemployment data column for all countries
unemp_2021 = unemp['SL.UEM.TOTL.ZS']
# The following line is no longer needed as the index of unemp_2021 is already correct (country codes)
# unemp_2021.index = [i for i in unemp_2021.index]

merged = country_risk.copy()
merged['unemployment'] = merged['country'].map(unemp_2021)
merged = merged.dropna()

X = sm.add_constant(merged['total_risk_score'])
y = merged['unemployment']
model2 = sm.OLS(y, X).fit()
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:           unemployment   R-squared:                       0.031
Model:                            OLS   Adj. R-squared:                 -0.039
Method:                 Least Squares   F-statistic:                    0.4405
Date:                Wed, 03 Jun 2026   Prob (F-statistic):              0.518
Time:                        21:59:12   Log-Likelihood:                -42.439
No. Observations:                  16   AIC:                             88.88
Df Residuals:                      14   BIC:                             90.42
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const               32.5122     38.339  